# Sarangi DDSP — Training (Colab)

Trains a **DDSP autoencoder** that learns **sarangi timbre** from ~10–20 min of
16 kHz mono audio, then exports the checkpoint to `model/sarangi_ddsp/`.

DDSP is **self-supervised**: it extracts f0 (CREPE) + loudness from raw audio and learns
to reconstruct the timbre. **No note labelling.** See `training/DATA_SPEC.md` and
`training/README.md`.

**Runtime:** free Colab GPU (`Runtime ▸ Change runtime type ▸ GPU`), ~2–4 hr.

---
### ⚠️ Dependency risk — read this
Magenta's DDSP repo (`github.com/magenta/ddsp`) was **archived in 2024**. The `ddsp` pip
package still installs but pins older TensorFlow. Pin **TensorFlow first, then ddsp**, and
**restart the runtime** after install. Known-good: `tensorflow==2.11.*` + `ddsp==3.7.0`.
Fallback: install ddsp from the (archived) git tag `v3.7.0`.


## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Install pinned dependencies
Pin TF **before** ddsp so the resolver doesn't silently upgrade TensorFlow.
**After this cell finishes, use `Runtime ▸ Restart runtime`, then continue from cell 2.**


In [ ]:
# Pin TensorFlow first...
!pip -q install "tensorflow==2.11.*"
# ...then DDSP + CREPE (f0). If the line below fails, use the git fallback under it.
!pip -q install "ddsp==3.7.0" "crepe==0.0.15"

# FALLBACK (uncomment) — install ddsp from the archived repo tag:
# !pip -q install "git+https://github.com/magenta/ddsp.git@v3.7.0"

# Audio utils used if you (re)run preprocessing here:
!pip -q install "librosa==0.10.1" "soundfile==0.12.1"

print("\n>>> Now: Runtime ▸ Restart runtime, then run from cell 2 onward. <<<")


## 2. Verify install (run AFTER restarting the runtime)

In [ ]:
import tensorflow as tf, ddsp, ddsp.training
print("tensorflow", tf.__version__)
print("ddsp", ddsp.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))


## 3. Get the dataset onto Colab

You need the **DDSP TFRecord** built from `training/data/processed/*.wav`. Two options:

**A. Build it here from processed clips** (if you only uploaded the 16 kHz mono WAVs), or
**B. Upload a TFRecord you already built** with `python training/preprocess.py --tfrecord`.

Pick ONE path below. First, choose where files live — Google Drive is easiest for large
data (survives disconnects).

In [ ]:
# --- Option: mount Google Drive (recommended for large data) ---
from google.colab import drive
drive.mount('/content/drive')

# TODO: point these at YOUR locations in Drive.
PROCESSED_DIR = '/content/drive/MyDrive/sarangi/processed'   # 16 kHz mono WAVs
TFRECORD_DIR  = '/content/drive/MyDrive/sarangi/tfrecord'    # output/location of TFRecord
import os
os.makedirs(TFRECORD_DIR, exist_ok=True)
print('processed:', PROCESSED_DIR)
print('tfrecord :', TFRECORD_DIR)


### 3A. Build the TFRecord here (CREPE f0 + loudness)
Run this only if you did **not** already build a TFRecord locally. Skip to 3B otherwise.

In [ ]:
# ddsp_prepare_tfrecord is a console script installed by the ddsp package.
# It runs CREPE (f0) + loudness extraction and shards the dataset.
TFRECORD_PATH = os.path.join(TFRECORD_DIR, 'sarangi.tfrecord')
!ddsp_prepare_tfrecord \
  --input_audio_filepatterns="{PROCESSED_DIR}/*.wav" \
  --output_tfrecord_path="{TFRECORD_PATH}" \
  --num_shards=10 \
  --alsologtostderr


### 3B. Or upload / locate a prebuilt TFRecord
If you ran `python training/preprocess.py --tfrecord` locally, upload the resulting
`sarangi.tfrecord*` shards to `TFRECORD_DIR` (via Drive or the Files pane) and set the
glob below.

In [ ]:
import glob
TFRECORD_PATH = os.path.join(TFRECORD_DIR, 'sarangi.tfrecord')
shards = sorted(glob.glob(TFRECORD_PATH + '*'))
assert shards, f'No TFRecord shards found at {TFRECORD_PATH}* — build (3A) or upload them.'
print(f'Found {len(shards)} shard(s):')
for s in shards: print(' ', s)


## 4. Training config (gin)

DDSP trains via `ddsp_run` with **gin** config files. We start from the stock
`solo_instrument.gin` autoencoder and override the dataset + a few knobs. This model
learns to reconstruct the sarangi from its own f0 + loudness — exactly the timbre transfer
we want at runtime.

`MODEL_DIR` is where checkpoints are written; we point it at Drive so training survives a
disconnect and can resume.

In [ ]:
MODEL_DIR = '/content/drive/MyDrive/sarangi/ddsp_model'   # TODO: your Drive path
os.makedirs(MODEL_DIR, exist_ok=True)

# TFRecord path pattern DDSP expects (the '*' matches all shards).
TFRECORD_PATTERN = TFRECORD_PATH + '*'
print('model dir:', MODEL_DIR)
print('data     :', TFRECORD_PATTERN)


## 5. Train (`ddsp_run`)

~2–4 hr on a free GPU. `num_steps=30000` is a reasonable target for ~10–20 min of audio;
watch the reconstruction loss and stop early if it plateaus. Training **resumes** from the
latest checkpoint in `MODEL_DIR` if you re-run after a disconnect.

The `--gin_param` lines override the dataset path and batch/step counts. Keep
`batch_size` modest to fit free-tier GPU memory.

In [ ]:
!ddsp_run \
  --mode=train \
  --alsologtostderr \
  --save_dir="{MODEL_DIR}" \
  --gin_file=models/solo_instrument.gin \
  --gin_file=datasets/tfrecord.gin \
  --gin_param="TFRecordProvider.file_pattern='{TFRECORD_PATTERN}'" \
  --gin_param="batch_size=16" \
  --gin_param="train_util.train.num_steps=30000" \
  --gin_param="train_util.train.steps_per_save=300" \
  --gin_param="trainers.Trainer.checkpoints_to_keep=3"


## 6. Quick sanity check (optional)
Load the trained model and reconstruct one batch to confirm it produces audio (not
silence/noise). This is a smoke test, not a quality bar — judge quality by ear at runtime.

In [ ]:
import ddsp, ddsp.training, gin
gin.parse_config_files_and_bindings(
    [ddsp.training.train_util.get_latest_operative_config(MODEL_DIR)], [])
# See ddsp colab demos for full inference; here we just confirm a checkpoint exists.
ckpt = ddsp.training.train_util.get_latest_chekpoint(MODEL_DIR) \
    if hasattr(ddsp.training.train_util, 'get_latest_chekpoint') else None
print('latest checkpoint:', ckpt)
!ls -la "{MODEL_DIR}"


## 7. Export checkpoint to `model/sarangi_ddsp/`

The runtime `sarangi_gen/synthesize.py` loads the checkpoint from **`model/sarangi_ddsp/`**
in the repo. Copy the trained checkpoint + the operative gin config there, then download /
commit it (plain commit if small, else Git LFS — see plan §11).

We copy the latest checkpoint files and `operative_config-*.gin` (needed to rebuild the
model graph at load time).

In [ ]:
import shutil, glob, os
EXPORT_DIR = '/content/sarangi_ddsp_export'    # then download this folder
os.makedirs(EXPORT_DIR, exist_ok=True)

# Checkpoint files: 'checkpoint', ckpt-*.index, ckpt-*.data-*, and the operative gin.
patterns = ['checkpoint', 'ckpt-*.index', 'ckpt-*.data-*', 'operative_config-*.gin']
copied = []
for pat in patterns:
    for f in glob.glob(os.path.join(MODEL_DIR, pat)):
        dst = os.path.join(EXPORT_DIR, os.path.basename(f))
        shutil.copy2(f, dst)
        copied.append(os.path.basename(f))
print('copied:', copied)

# Zip for download; unzip into the repo at model/sarangi_ddsp/
shutil.make_archive('/content/sarangi_ddsp', 'zip', EXPORT_DIR)
from google.colab import files
files.download('/content/sarangi_ddsp.zip')
print('\nUnzip into the repo at:  model/sarangi_ddsp/')


---
## If quality is poor (same-day fallback)
Per plan §9: if the model isn't usable by ~hour 5, don't block the ship. Loop the best
tempo-consistent **solo** passage through the runtime `sarangi_gen/post.py` (still gives
exact loop length + seam + cache-key name) so the instrument ships today, and keep
improving the model afterward.
